In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

# 1. Load Dataset từ file CSV có sẵn trong thư mục data
# Đường dẫn tương đối từ thư mục notebooks ra thư mục data
csv_path = '../data/custom_dataset/breast_cancer.csv'

if os.path.exists(csv_path):
    print(f"Đang đọc dữ liệu từ: {csv_path}")
    df = pd.read_csv(csv_path)
else:
    print("Không tìm thấy file CSV, đang sử dụng dữ liệu mặc định từ sklearn...")
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target

# Hiển thị 5 dòng đầu để kiểm tra
print(f"Kích thước bộ dữ liệu: {df.shape}")
display(df.head())

# Tách Features (X) và Label (y)
# Giả định cột cuối cùng là nhãn (target). Nếu tên cột target khác, bạn sửa lại nhé.
X = df.iloc[:, :-1] # Lấy tất cả cột trừ cột cuối
y = df.iloc[:, -1]  # Lấy cột cuối cùng làm nhãn
class_names = [str(c) for c in np.unique(y)]
feature_names = X.columns.tolist()

print("\nFeatures:", feature_names[:5], "...") # In thử vài feature
print("Target classes:", class_names)

In [ ]:
# Danh sách các tỷ lệ train
ratios = [0.4, 0.6, 0.8, 0.9]

for ratio in ratios:
    test_size = 1.0 - ratio
    print(f"\n{'='*20} TỶ LỆ TRAIN: {int(ratio*100)}% - TEST: {int(test_size*100 + 0.1)}% {'='*20}")
    
    # Chia dữ liệu (Stratified shuffle split) [cite: 30]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, shuffle=True, random_state=42
    )
    
    # Huấn luyện mô hình (sử dụng Entropy làm tiêu chí) [cite: 37]
    clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
    clf.fit(X_train, y_train)
    
    # Dự đoán
    y_pred = clf.predict(X_test)
    
    # Báo cáo kết quả [cite: 62]
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Vẽ cây minh họa (Chỉ vẽ độ sâu = 2 để dễ nhìn, phục vụ báo cáo)
    plt.figure(figsize=(12, 6))
    plot_tree(clf, feature_names=feature_names, class_names=class_names, filled=True, max_depth=2)
    plt.title(f"Decision Tree (Depth=2 View) - Train {int(ratio*100)}%")
    plt.show()

In [ ]:
print(f"\n{'='*20} KHẢO SÁT ĐỘ SÂU (Train 80/20) {'='*20}")

# Cố định tập train/test 80/20 [cite: 100]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42
)

# Các giá trị độ sâu cần khảo sát [cite: 103]
depths = [2, 3, 4, 5, 6, 7, None]
results = []

for depth in depths:
    clf = DecisionTreeClassifier(criterion='entropy', max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    depth_label = str(depth) if depth is not None else "None"
    results.append((depth_label, acc))
    
    print(f"Max Depth: {depth_label} -> Accuracy: {acc:.4f}")

# Vẽ biểu đồ đường thể hiện độ chính xác theo độ sâu [cite: 107]
df_res = pd.DataFrame(results, columns=["Max Depth", "Accuracy"])
plt.figure(figsize=(10, 5))
plt.plot(df_res["Max Depth"], df_res["Accuracy"], marker='o', color='purple', linestyle='-')
plt.title("Ảnh hưởng của độ sâu (Max Depth) đến Độ chính xác")
plt.xlabel("Độ sâu tối đa (Max Depth)")
plt.ylabel("Độ chính xác (Accuracy)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# In bảng kết quả để copy vào báo cáo
display(df_res)